In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료!')

---

## 1단계: 학습 데이터 (기준 분포) 로드

모델은 이 데이터의 분포를 기반으로 학습했습니다.  
운영 데이터가 이 분포에서 벗어나면 모델 성능이 떨어질 수 있습니다.

In [ ]:
# 여기에 코드를 작성하세요


---

## 2단계: 예측 로그 (운영 데이터) 로드

차시 27에서 설계한 예측 로그가 3개월간 쌓인 결과입니다.  
실무에서는 CloudWatch Logs에서 쿼리하여 CSV로 추출합니다.

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


---

## 3단계: PSI로 드리프트 감지

**PSI (Population Stability Index)**
- 금융권(NICE, KCB)에서 실제로 사용하는 드리프트 지표
- 두 분포의 차이를 숫자 하나로 요약

| PSI | 판정 | 조치 |
|-----|------|------|
| < 0.1 | 안정 | 조치 불필요 |
| 0.1 ~ 0.25 | 주의 | 모니터링 강화 |
| > 0.25 | **드리프트** | **재학습 검토** |

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


---

## 4단계: 재학습 판단 & 실행

드리프트가 감지되었으므로 재학습을 진행하고, 기존 모델과 성능을 비교합니다.

### 실무 흐름

```
PSI > 0.25 감지 → 알림(Slack 등) → 재학습 실행 → 성능 비교 → 더 좋으면 재배포
```

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


---

## 현업에서는 드리프트 감지를 얼마나 자주 하는가?

이 실습에서는 수동으로 PSI를 계산했지만, 실무에서는 **자동으로 주기적 실행**합니다.
그런데 그 주기는 도메인마다 다릅니다.

### 도메인별 드리프트 감지 주기

| 도메인 | 감지 주기 | 이유 |
|--------|----------|------|
| **대출/보험** (우리 프로젝트) | 주 1회 ~ 월 1회 | 고객 특성이 천천히 변함, 규제 산업이라 신중해야 함 |
| **이커머스/광고** | 매일 | 시즌/트렌드에 따라 사용자 행동이 빠르게 변함 |
| **사기 탐지** | 시간 단위 | 공격 패턴이 갑자기 바뀔 수 있음 |
| **제조/IoT 센서** | 주 1회 | 센서 드리프트가 서서히 발생 |

### 왜 예측할 때마다 하지 않는가?

- PSI는 **두 집단의 분포**를 비교하는 통계 지표이므로, 데이터가 충분히 쌓여야 의미가 있다
- 요청 1건으로는 분포를 판단할 수 없고, 최소 수백~수천 건을 모아야 통계적으로 유의미하다
- 예측할 때마다 계산하면 불필요한 연산 비용이 발생한다

### 우리 프로젝트에 적용하면

```
매주 월요일 새벽 2시 (cron 자동 실행)
    |
    v
PSI 계산 스크립트 실행
    |
    +-- PSI < 0.1    --> 로그만 남기고 종료 (안정)
    +-- PSI 0.1~0.25 --> Slack 알림: "주의, 모니터링 강화"
    +-- PSI > 0.25   --> Slack 알림: "드리프트 감지! 재학습 검토 필요"
                           |
                           v
                      사람이 판단 -> 재학습 노트북(27_3) 실행
```

### cron 설정 예시

```bash
# 매주 월요일 새벽 2시에 PSI 체크
crontab -e
0 2 * * 1 python /app/scripts/check_drift.py >> /var/log/drift.log 2>&1
```

### 핵심 정리

> **드리프트 감지는 자동, 재학습 결정은 사람이 한다.**
>
> 예측할 때마다 하는 게 아니라, 로그를 모아서 주기적으로 비교한다.
> 주기는 데이터가 얼마나 빨리 변하느냐에 따라 결정한다.

---

## 정리

### 이번 실습에서 한 것

1. **학습 데이터 vs 예측 로그 비교**: 3개월간 쌓인 로그에서 분포 변화 확인
2. **PSI 계산**: 피처별 드리프트 수치 측정 (금융권 실무 표준)
3. **드리프트 판정**: PSI > 0.25인 피처 식별
4. **재학습 → 성능 비교**: 새 데이터 포함하여 재학습 후 AUC 비교 → 배포 판단

### MLOps 운영 자동화 전체 사이클

```
코드 Push → CI/CD → 배포 → 예측 로그 수집 → 드리프트 감지 → 재학습 → 재배포
```

이것이 **"MLOps 기반 운영 자동화"**의 전체 그림입니다.

